<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 10B: *Fire Damage Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
train_X = pd.read_csv('../data/processed/train_train_X.csv')
train_y = pd.read_csv('../data/processed/train_y_damage.csv').squeeze()  # Load as Series      
train_details = pd.read_csv('../data/processed/train_details.csv')

test_X = pd.read_csv('../data/processed/test_X.csv')
test_y = pd.read_csv('../data/processed/test_y_damage.csv').squeeze()  # Load as Series      
test_details = pd.read_csv('../data/processed/test_details.csv')

with open('../data/processed/model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('../data/processed/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/train_train_X.csv'

## Build Models

In [ ]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build Final tuned models
damage_xgb = xgb.XGBClassifier(**XGB_parameters)
damage_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

In [ ]:
models = {
    "XGB": damage_xgb, 
    "RF": damage_rf
}

## SHAP

In [ ]:
xgb_top = get_shap(damage_xgb, train_X, train_y, test_X, test_y)
rf_top = get_shap(damage_rf, train_X, train_y, test_X, test_y)

In [ ]:
weather_features = []

drop = ['fire_count','fire_count 3 Day Sum','fire_count 7 Day Sum','fire_count 30 Day Sum',
       'road_density_x_forest_percent','power_line_density_x_log_total_housing']
weather_sets = ['Water Demand','Water Supply','Water Supply Indexes','Fire Danger','Interactions','Wind Slope',
                              'Lag 3 Day Features','Lag 7 Day Features','Lag 30 Day Features']

for weather_set in weather_sets:
    for feature in feature_sets[weather_set]:
        if feature not in drop:
            weather_features.append(feature)

In [ ]:
top_xgb_weather = xgb_top.loc[xgb_top['Feature'].isin(weather_features)]
top_xgb_weather[:5]

In [ ]:
top_rf_weather = rf_top.loc[rf_top['Feature'].isin(weather_features)]
top_rf_weather[:5]

In [ ]:
common_ranked = (
    rf_top[:20][['Feature', 'SHAP Importance']]
    .rename(columns={'SHAP Importance': 'RF_Importance'})
    .merge(
        xgb_top[:20][['Feature', 'SHAP Importance']]
        .rename(columns={'SHAP Importance': 'XGB_Importance'}),
        on='Feature',
        how='inner'
    )
    .sort_values('XGB_Importance', ascending=False)
    .reset_index(drop=True)
)

In [ ]:
xgb_top[:10]

In [ ]:
rf_top[:10]

In [ ]:
common_ranked

In [ ]:
explainer = shap.TreeExplainer(damage_xgb)
shap_values = explainer.shap_values(train_X)

In [ ]:
shap_class = shap_values[:, :, 4]

In [ ]:
shap_class.shape == train_X.shape

In [ ]:
shap.dependence_plot(
    "Solar Radiation 7 Day Sum",
    shap_class,
    train_X,
    interaction_index=None
)

In [ ]:
shap.dependence_plot(
    "SPEI 90-Day",
    shap_class,
    train_X,
    interaction_index=None
)

## Set Ablation

In [ ]:
Ablation_single_column = ablation(models, train_X, train_y, train_X, train_y, feature_sets, best_strategy, single_set = False)

In [ ]:
# Assume your table is in a DataFrame called df
# Filter out the full runs
full_model = Ablation_single_column[Ablation_single_column['Test Set'] == 'Full'][['Model', 'Macro F1']].rename(columns={'Macro F1': 'Full Macro F1'})

# Merge back to the main dataframe on 'Model'
pivot_ablation = Ablation_single_column.merge(full_model, on='Model', how='left')

# Drop columns you don’t need for now
pivot_ablation = pivot_ablation.drop(columns=['Weighted F1', 'High Risk Recall'])

largest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] > .2]
smallest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] < 0]

In [ ]:
largest_drops

In [ ]:
smallest_drops

In [ ]:
Ablation_single_column